In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("EmployeeSalaryAnalytics").getOrCreate()

employee_data = [
(101, "Sravan", "Data Engineer", "IT", 75000, "Hyderabad", 28, "2021-05-10", "Male"),
(102, "Ravi", "Software Engineer", "IT", 68000, "Bangalore", 30, "2020-03-15", "Male"),
(103, "Priya", "Data Analyst", "Analytics", 62000, "Chennai", 26, "2022-01-12", "Female"),
(104, "Kiran", "Manager", "HR", 90000, "Mumbai", 35, "2018-07-19", "Male"),
(105, "Anjali", "HR Executive", "HR", 45000, "Pune", 24, "2023-02-20", "Female"),
(106, "Vikram", "Data Scientist", "Analytics", 98000, "Delhi", 32, "2019-11-25", "Male"),
(107, "Sneha", "Developer", "IT", 71000, "Hyderabad", 27, "2021-08-17", "Female"),
(108, "Rahul", "Tester", "QA", 55000, "Chennai", 29, "2020-06-10", "Male"),
(109, "Meena", "QA Lead", "QA", 83000, "Bangalore", 33, "2017-09-14", "Female"),
(110, "Arjun", "Support Engineer", "Support", 50000, "Pune", 31, "2022-04-11", "Male")
]

columns = [
    "emp_id",
    "name",
    "designation",
    "department",
    "salary",
    "city",
    "age",
    "joining_date",
    "gender"
]

emp_df = spark.createDataFrame(employee_data, columns)

emp_df.show()

+------+------+-----------------+----------+------+---------+---+------------+------+
|emp_id|  name|      designation|department|salary|     city|age|joining_date|gender|
+------+------+-----------------+----------+------+---------+---+------------+------+
|   101|Sravan|    Data Engineer|        IT| 75000|Hyderabad| 28|  2021-05-10|  Male|
|   102|  Ravi|Software Engineer|        IT| 68000|Bangalore| 30|  2020-03-15|  Male|
|   103| Priya|     Data Analyst| Analytics| 62000|  Chennai| 26|  2022-01-12|Female|
|   104| Kiran|          Manager|        HR| 90000|   Mumbai| 35|  2018-07-19|  Male|
|   105|Anjali|     HR Executive|        HR| 45000|     Pune| 24|  2023-02-20|Female|
|   106|Vikram|   Data Scientist| Analytics| 98000|    Delhi| 32|  2019-11-25|  Male|
|   107| Sneha|        Developer|        IT| 71000|Hyderabad| 27|  2021-08-17|Female|
|   108| Rahul|           Tester|        QA| 55000|  Chennai| 29|  2020-06-10|  Male|
|   109| Meena|          QA Lead|        QA| 83000|Ban

**1.Find Top 3 Highest Salaries Department-Wise**

In [2]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

top3_salary = emp_df.withColumn("rank", dense_rank().over(window_spec)) \
                    .filter(col("rank") <= 3)

top3_salary.select(
    "department",
    "emp_id",
    "name",
    "designation",
    "salary",
    "rank"
).show()

+----------+------+------+-----------------+------+----+
|department|emp_id|  name|      designation|salary|rank|
+----------+------+------+-----------------+------+----+
| Analytics|   106|Vikram|   Data Scientist| 98000|   1|
| Analytics|   103| Priya|     Data Analyst| 62000|   2|
|        HR|   104| Kiran|          Manager| 90000|   1|
|        HR|   105|Anjali|     HR Executive| 45000|   2|
|        IT|   101|Sravan|    Data Engineer| 75000|   1|
|        IT|   107| Sneha|        Developer| 71000|   2|
|        IT|   102|  Ravi|Software Engineer| 68000|   3|
|        QA|   109| Meena|          QA Lead| 83000|   1|
|        QA|   108| Rahul|           Tester| 55000|   2|
|   Support|   110| Arjun| Support Engineer| 50000|   1|
+----------+------+------+-----------------+------+----+



**2.Find Average Salary City-Wise**

In [3]:
city_avg_salary = emp_df.groupBy("city") \
    .agg(round(avg("salary"), 2).alias("average_salary")) \
    .orderBy(col("average_salary").desc())

city_avg_salary.show()

+---------+--------------+
|     city|average_salary|
+---------+--------------+
|    Delhi|       98000.0|
|   Mumbai|       90000.0|
|Bangalore|       75500.0|
|Hyderabad|       73000.0|
|  Chennai|       58500.0|
|     Pune|       47500.0|
+---------+--------------+



**3.Categorize Employees into Salary Bands**

In [4]:
salary_band = emp_df.withColumn(
    "salary_band",
    when(col("salary") >= 90000, "High")
    .when((col("salary") >= 60000) & (col("salary") < 90000), "Medium")
    .otherwise("Low")
)

salary_band.select(
    "emp_id",
    "name",
    "salary",
    "salary_band"
).show()

+------+------+------+-----------+
|emp_id|  name|salary|salary_band|
+------+------+------+-----------+
|   101|Sravan| 75000|     Medium|
|   102|  Ravi| 68000|     Medium|
|   103| Priya| 62000|     Medium|
|   104| Kiran| 90000|       High|
|   105|Anjali| 45000|        Low|
|   106|Vikram| 98000|       High|
|   107| Sneha| 71000|     Medium|
|   108| Rahul| 55000|        Low|
|   109| Meena| 83000|     Medium|
|   110| Arjun| 50000|        Low|
+------+------+------+-----------+



**4.Generate Yearly Salary Report**

In [5]:
yearly_report = emp_df.withColumn(
    "joining_year",
    year(to_date(col("joining_date"), "yyyy-MM-dd"))
)

salary_report = yearly_report.groupBy("joining_year") \
    .agg(
        count("*").alias("employee_count"),
        sum("salary").alias("total_salary"),
        round(avg("salary"), 2).alias("average_salary"),
        min("salary").alias("min_salary"),
        max("salary").alias("max_salary")
    ) \
    .orderBy("joining_year")

salary_report.show()

+------------+--------------+------------+--------------+----------+----------+
|joining_year|employee_count|total_salary|average_salary|min_salary|max_salary|
+------------+--------------+------------+--------------+----------+----------+
|        2017|             1|       83000|       83000.0|     83000|     83000|
|        2018|             1|       90000|       90000.0|     90000|     90000|
|        2019|             1|       98000|       98000.0|     98000|     98000|
|        2020|             2|      123000|       61500.0|     55000|     68000|
|        2021|             2|      146000|       73000.0|     71000|     75000|
|        2022|             2|      112000|       56000.0|     50000|     62000|
|        2023|             1|       45000|       45000.0|     45000|     45000|
+------------+--------------+------------+--------------+----------+----------+



**5.Find Employees Earning Above Department Average**

In [6]:
dept_avg = emp_df.groupBy("department") \
    .agg(avg("salary").alias("dept_avg_salary"))

above_avg = emp_df.join(dept_avg, "department") \
    .filter(col("salary") > col("dept_avg_salary")) \
    .select(
        "department",
        "emp_id",
        "name",
        "salary",
        round(col("dept_avg_salary"), 2).alias("dept_avg_salary")
    )

above_avg.show()

+----------+------+------+------+---------------+
|department|emp_id|  name|salary|dept_avg_salary|
+----------+------+------+------+---------------+
|        HR|   104| Kiran| 90000|        67500.0|
|        IT|   101|Sravan| 75000|       71333.33|
| Analytics|   106|Vikram| 98000|        80000.0|
|        QA|   109| Meena| 83000|        69000.0|
+----------+------+------+------+---------------+

